# 05 — Model Experiments & Research Decisions

The modeling stage developed iteratively rather than following a single predetermined model.

The experiments below separate:

1. what was actually explored during the real project;
2. what was learned from those experiments;
3. what is reproduced publicly with synthetic data.

No production dataset or production model artifact is included here.

## 1. Experimental path

### Initial model exploration

The first modeling attempts included:

- Random Forest
- XGBoost
- LightGBM
- Dask-based training

The main constraint was scale: the working dataset contained millions of loan records and a large mixture of categorical and numerical variables.

### Shift toward interpretable GLM

Later experiments moved toward H2O GLM / logistic regression.

This was driven by the need to understand coefficient direction, statistical significance and the effect of different feature representations rather than treating prediction as a pure benchmark problem.

### Representation experiments

Several versions were tested:

- raw numeric variables;
- categorized variables;
- missing-value categories;
- one-hot encoded categorical variables;
- reduced categorical representations;
- alternative geographic/product groupings.

### Large-scale experiments

The later iterations also explored:

- sparse/high-cardinality feature spaces;
- stratified sampling;
- observation weighting;
- H2O memory configuration;
- Parquet/chunked data handling;
- removal of constant or problematic columns.

## 2. What changed between experiments

### Early boosting experiments

Boosting models were useful as nonlinear benchmarks.

They showed that the problem was not limited to a simple linear boundary, but they did not remove the need to understand the feature representation itself.

### H2O GLM experiments

GLM provided a more transparent modeling framework.

The experiments showed that model behavior changed materially when:

- missing values were represented differently;
- numeric variables were categorized;
- geographic/product categories were encoded differently;
- the feature space became very sparse.

### Feature-space problem

Some one-hot representations produced very large numbers of sparse categorical columns.

H2O also reported constant/bad columns during GLM fitting in the real experiments. This became evidence that preserving every granular category was not automatically the best representation.

### Final direction

The project therefore moved toward an **interpretable logistic/GLM-centered approach**, while retaining tree-based models as useful benchmarks.

## 3. A representative decision log

| Question | Experiment | Observation | Decision |
|---|---|---|---|
| Can nonlinear models capture additional structure? | XGBoost / LightGBM / Random Forest | Useful benchmark families | Keep as benchmarks |
| Can the problem be modeled transparently? | H2O GLM / logistic | Coefficients and diagnostics are directly inspectable | Keep as primary direction |
| How should missing values be represented? | Separate missing vs non-missing datasets | Representation changed model behavior | Treat missingness as a modeling decision |
| Should continuous variables remain raw? | Raw vs categorized variables | Categorization changed fit and interpretability | Compare representations |
| Can all geographic detail be retained? | Fine-grained one-hot encoding | Very wide/sparse feature space | Investigate grouping |
| Can the large dataset be modeled directly? | H2O / Dask / chunked data | Memory and computational constraints became part of the problem | Use scalable processing strategies |
| Can categories be preserved across splits? | Stratified/grouped representations | Rare category support differs between partitions | Control category structure before modeling |

## 4. Public reproducible baseline

The following experiment uses the synthetic dataset and keeps the model comparison deliberately small.

The aim is to reproduce the **decision framework**, not the confidential production results.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

DATA_PATH = Path("../data/synthetic/synthetic_early_modeling_base.csv")
df = pd.read_csv(DATA_PATH)

freq_days = {"Weekly": 7, "Bi-weekly": 14, "Monthly": 28}

df["pass_due_cycle_ratio"] = (
    df["early_max_overdue_days"]
    / df["frequency_name"].map(freq_days)
)

df["missed_installment_proportion"] = (
    df["early_missed_installment_count"]
    / df["prediction_installment"].clip(lower=1)
)

df["overdue_amount_proxy"] = (
    df["early_missed_installment_count"]
    * df["installment_amount"]
)

df["overdue_proportion"] = (
    df["overdue_amount_proxy"]
    / (
        df["prediction_installment"].clip(lower=1)
        * df["installment_amount"]
    )
).clip(0, 1)

target = "is_good_or_bad"

features = [
    "frequency_name",
    "product_group",
    "sector",
    "region",
    "disbursed_amount",
    "installment_amount",
    "interest_rate",
    "prior_loan_count",
    "early_missed_installment_count",
    "missed_installment_proportion",
    "early_max_consecutive_missed",
    "early_max_overdue_days",
    "pass_due_cycle_ratio",
    "early_recovery_delay_cycles",
    "overdue_proportion",
]

X = df[features].copy()
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

## 5. Logistic regression baseline

In [ ]:
numeric_features = [
    "disbursed_amount",
    "installment_amount",
    "interest_rate",
    "prior_loan_count",
    "early_missed_installment_count",
    "missed_installment_proportion",
    "early_max_consecutive_missed",
    "early_max_overdue_days",
    "pass_due_cycle_ratio",
    "early_recovery_delay_cycles",
    "overdue_proportion",
]

categorical_features = [
    "frequency_name",
    "product_group",
    "sector",
    "region",
]

preprocess = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        numeric_features,
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]),
        categorical_features,
    ),
])

logistic = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    )),
])

logistic.fit(X_train, y_train)

p_logit = logistic.predict_proba(X_test)[:, 1]

logit_results = {
    "ROC-AUC": roc_auc_score(y_test, p_logit),
    "PR-AUC": average_precision_score(y_test, p_logit),
}

logit_results

## 6. Decision tree benchmark

In [ ]:
tree_preprocess = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]),
        numeric_features,
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]),
        categorical_features,
    ),
])

tree = Pipeline([
    ("preprocess", tree_preprocess),
    ("model", DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=42,
    )),
])

tree.fit(X_train, y_train)

p_tree = tree.predict_proba(X_test)[:, 1]

tree_results = {
    "ROC-AUC": roc_auc_score(y_test, p_tree),
    "PR-AUC": average_precision_score(y_test, p_tree),
}

tree_results

## 7. Compare the public baselines

In [ ]:
comparison = pd.DataFrame({
    "Logistic Regression": logit_results,
    "Decision Tree": tree_results,
}).T

comparison.sort_values("PR-AUC", ascending=False)

## 8. What this comparison is for

The point of the benchmark is not to declare a universal winner.

The production research problem required balancing predictive behavior with:

- interpretability;
- early feature availability;
- high-cardinality categorical structure;
- computational feasibility;
- statistical diagnostics.

Those constraints are why model selection was treated as a research decision rather than a leaderboard exercise.

## 9. Experiments not yet reproduced publicly

The real project also considered or tested additional approaches including:

- alternative LightGBM/XGBoost configurations;
- H2O GLM variants;
- different missingness treatments;
- observation weighting;
- high-cardinality geographic/product grouping;
- sparse one-hot representations;
- chunked/Parquet processing;
- tabular deep-learning approaches such as TabTransformer;
- alternative linear optimization approaches such as SGDClassifier.

The last two belong to future methodological work rather than completed experiments.

## 10. Takeaway

The modeling process was iterative:

**model family → representation → computational constraint → diagnostic result → revised representation → new model experiment**

That iteration is an important part of the study.

The next stage can extend the comparison with gradient boosting, imbalance strategies, probability thresholds and model interpretation.